# non-diff-fn-wrap — faded example 1: Three-gate AND with is_differentiable flag

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `non-diff-fn-wrap`. The last cell reports your progress on the `Backprop: non-differentiable fn wrap` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: non-differentiable fn wrap` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`non-diff-fn-wrap`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "non-diff-fn-wrap"
DD_SUBTOPIC = "Backprop: non-differentiable fn wrap"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The `wrap_forward_fn` factory builds a wrapper that gates gradient tracking through three ANDed conditions: global tracking enabled, the op is marked differentiable, and at least one input has `requires_grad=True`. Non-differentiable ops pass `is_differentiable=False`, which causes the AND to short-circuit to False and prevents Recipe creation.

## Faded exercise 1

Complete `wrap_forward_fn`. The unboxing and forward call are provided. Fill in the `requires_grad` computation using the three-gate AND: `grad_tracking_enabled AND is_differentiable AND any(tracked inputs)`.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch

class Recipe:
    def __init__(self, func, args, kwargs, parents):
        self.func = func; self.args = args
        self.kwargs = kwargs; self.parents = parents

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array; self.requires_grad = requires_grad; self.recipe = None

grad_tracking_enabled = True

def wrap_forward_fn(fwd_fn, is_differentiable: bool = True):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_arr = fwd_fn(*raw_args, **kwargs)
        requires_grad = (
            grad_tracking_enabled
            and is_differentiable
            and any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
        )
        out = MiniTensor(out_arr, requires_grad)
        if requires_grad:
            parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func

mul_wrap = wrap_forward_fn(torch.mul)
eq_wrap  = wrap_forward_fn(torch.eq, is_differentiable=False)
x = MiniTensor(torch.tensor([2.0, 3.0]), requires_grad=True)
print(mul_wrap(x, x).requires_grad)  # True
print(eq_wrap(x, x).requires_grad)   # False


def _test():
    import torch
    x = MiniTensor(torch.tensor([2.0, 3.0]), requires_grad=True)
    y = MiniTensor(torch.tensor([2.0, 4.0]), requires_grad=True)
    mul_wrap = wrap_forward_fn(torch.mul)
    eq_wrap  = wrap_forward_fn(torch.eq, is_differentiable=False)
    r_mul = mul_wrap(x, y)
    r_eq  = eq_wrap(x, y)
    assert r_mul.requires_grad is True,  'differentiable op with tracked inputs should have requires_grad=True'
    assert r_mul.recipe is not None,     'differentiable op should have a Recipe'
    assert r_eq.requires_grad is False,  'non-differentiable op must have requires_grad=False'
    assert r_eq.recipe is None,          'non-differentiable op must have recipe=None'


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch

class Recipe:
    def __init__(self, func, args, kwargs, parents):
        self.func = func; self.args = args
        self.kwargs = kwargs; self.parents = parents

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array; self.requires_grad = requires_grad; self.recipe = None

grad_tracking_enabled = True

def wrap_forward_fn(fwd_fn, is_differentiable: bool = True):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_arr = fwd_fn(*raw_args, **kwargs)
        requires_grad = (
            grad_tracking_enabled
            and is_differentiable
            and any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
        )
        out = MiniTensor(out_arr, requires_grad)
        if requires_grad:
            parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func

mul_wrap = wrap_forward_fn(torch.mul)
eq_wrap  = wrap_forward_fn(torch.eq, is_differentiable=False)
x = MiniTensor(torch.tensor([2.0, 3.0]), requires_grad=True)
print(mul_wrap(x, x).requires_grad)  # True
print(eq_wrap(x, x).requires_grad)   # False
```
</details>